# Qwen3-VL 补训 v3：NEU-DET 钢材表面缺陷定位（只训答案 + 2 epochs）

背景：FabricSpotDefect 补训 v2 模式坍缩（mean IoU 0.0173，低于基线 0.1084；151/152 张输出 9 个框、134 张所有框一模一样）。
诊断结论：织物斑点目标太小（框中位数只占图 0.46%），模型学不到定位信号；同时 3 epochs 过拟合了重复模式。

本版改动（其余与 v2 完全相同，便于归因）：
1. **换数据集 NEU-DET**：1797 张钢材表面缺陷，6 类；框中位数占图 11.82%（斑点的约 26 倍），0% 的框小于 30×30；
   框中心分布散开（仅 11.9% 落在中心 20% 区域内），无位置捷径；6 类叠加抽查框与可见缺陷对得上。
2. **2 epochs**（v2 是 3）：防坍缩。训练集 1,437 张图、1,534 个（图，类别）样本（95 张图含 2 类、2 张含 3 类，混类图按类别拆成多条样本），等效 batch 16，约 192 步，T4 上约 1 小时。
3. **训练前先验数据**：逐类计数、中心分布、去重、存 6 张叠加图，任一项异常即停，不烧 GPU。
4. **EOS 自查**：kept labels 必须以 `<|im_end|>` 结尾，否则模型学不会停（v2 的失败模式）。
5. **只存适配器**（`neudet-adapter-v3`），不合并——评估本直接加载基座 + 适配器。
6. **2026-11-24 修复（首跑失败后）**：① transformers 锁定 4.57.1——首跑时 `pip install -U` 拉到最新 5.x，
   其 Qwen3-VL 视觉路径与本管线不兼容，训练第一步即报 `hidden_states + pos_embeds` 尺寸不匹配；
   ② **修掉 v2 就有的 pixel_values[0] 取首补丁 bug**：处理器返回的 pixel_values 已是切块后的 (256, 1536)、无 batch 维，
   `[0]` 只剩第一个补丁，模型看到的是同一补丁在 256 个位置上重复——视觉信号基本是空的，
   这很可能就是 v2 只学会格式、定位坍缩到众数框的根因；自查 cell 已加补丁数与 grid 一致性断言。
7. **2026-11-24 绕过 HF 429（V3–V6 均死于此）**：Kaggle 出口 IP 匿名下载 NEU-DET 被 Hugging Face 限流，
   三次重跑换了三个出口 IP 全撞 429。改走 Kaggle 输入：数据集 zip（neu-det-corrected.zip，本地同名文件）
   随笔记本挂为输入，首 cell 先解压它，HF 下载只作后备。

其余与 v2 相同：基座 Qwen/Qwen3-VL-2B-Instruct、QLoRA NF4 r=8 alpha=16、lr 1e-4、等效 batch 16、损失只算答案。
验收线（NEU-DET 测试集 180 张，同口径评估）：格式合规率接近 100%，且 Acc@IoU>=0.5 高于基线。

In [ ]:
!pip install -q -U "transformers==4.57.1" accelerate peft bitsandbytes huggingface_hub
import os
# ---- Dataset source: Kaggle input first, HF as fallback ----
# HF 429: anonymous downloads from Kaggle egress IPs are rate-limited
# (V3-V6 all died here). The dataset is attached as Kaggle input.
# Probe (2026-11-24) found the real mount path is nested:
# /kaggle/input/datasets/gmy723/neu-det-corrected (extracted files,
# NOT a zip) - the flat /kaggle/input/neu-det-corrected guess was wrong.
DATA_ROOT = None
for _cand in ['/kaggle/input/datasets/gmy723/neu-det-corrected',
              '/kaggle/input/neu-det-corrected']:
    _z = os.path.join(_cand, 'neu-det-corrected.zip')
    if os.path.isfile(_z):
        import zipfile
        DATA_ROOT = '/kaggle/working/neudet'
        print('unzipping dataset from', _z)
        zipfile.ZipFile(_z).extractall(DATA_ROOT)
        break
    if os.path.isdir(os.path.join(_cand, 'images')):
        DATA_ROOT = _cand
        break
if DATA_ROOT is None:
    from huggingface_hub import snapshot_download
    DATA_ROOT = snapshot_download(repo_id='KeenForgeAI/NEU-DET-corrected', repo_type='dataset')
print('NEU-DET ->', DATA_ROOT)
import transformers
assert transformers.__version__ == '4.57.1', f"pinned transformers 4.57.1 required, got {transformers.__version__}"
print('transformers', transformers.__version__, '(pinned: latest 5.x breaks the Qwen3-VL vision path)')
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os, glob, random, json
from PIL import Image

CLS_CN = {'crazing': '裂纹', 'inclusion': '夹杂物', 'patches': '斑块', 'pitted_surface': '麻点', 'rolled-in_scale': '氧化皮', 'scratches': '划痕'}
TEMPLATES = ['请识别图片中的钢材表面缺陷（{cn}），并用边界框标出所有位置。', '找出图片中所有的钢材表面缺陷（{cn}），用边界框标出。', '定位图片中的钢材表面缺陷（{cn}），给出每个缺陷的边界框。', '检测图片中的钢材表面缺陷（{cn}），并框出它们的位置。', '请标出这张钢材图片中所有缺陷（{cn}）的位置。', '在这张图片中找到钢材表面缺陷（{cn}），用边界框标出。', '请用边界框框出图片中的所有钢材表面缺陷（{cn}）。', '把图片中钢材表面缺陷（{cn}）的位置用框圈出来。', '用矩形框标出图片里所有的钢材表面缺陷（{cn}）。', '请圈出图中所有的钢材表面缺陷（{cn}）。', '找出图中的钢材表面缺陷（{cn}）并用方框标出。', '检测这张钢材图片，定位其中的缺陷（{cn}）。', '对图片进行缺陷检测，标出所有钢材表面缺陷（{cn}）。', '请检测图片中的钢材表面缺陷（{cn}）并给出边界框。', '执行钢材表面缺陷检测，标出图中所有{cn}的位置。', '检查这张图片，找出其中的钢材表面缺陷（{cn}）并定位。', '识别图片中的钢材表面缺陷（{cn}），给出边界框坐标。', '请识别图中所有的钢材表面缺陷（{cn}）并标出位置。', '这张图片里有哪些钢材表面缺陷（{cn}）？请用边界框标出。', '识别并定位图片中的钢材表面缺陷（{cn}）。', '钢材表面缺陷（{cn}）在图片的什么位置？请用边界框标出。', '图中的钢材表面缺陷（{cn}）在哪里？请框出来。', '请指出图片中钢材表面缺陷（{cn}）的位置。', '请标出图片中每一个钢材表面缺陷（{cn}）。', '不要漏掉任何一个，标出图片中所有的钢材表面缺陷（{cn}）。', '图中共有几个钢材表面缺陷（{cn}）？请全部框出来。', '列出图片中全部钢材表面缺陷（{cn}）的边界框。', '标出缺陷（{cn}）。', '定位缺陷（{cn}）。', '框出所有{cn}。']
assert len(TEMPLATES) == 30

TRAIN_LBL = os.path.join(DATA_ROOT, 'labels', 'train')
TRAIN_IMG = os.path.join(DATA_ROOT, 'images', 'train')
CLASSES = open(os.path.join(DATA_ROOT, 'classes.txt')).read().split()
WORK = '/kaggle/working'
ADAPTER_DIR = os.path.join(WORK, 'neudet-adapter-v3')

def build_samples(seed=42):
    rng = random.Random(seed)
    samples, skipped = [], {"no_image": 0, "no_box": 0, "dup_box": 0, "bad_image": 0}
    for lp in sorted(glob.glob(os.path.join(TRAIN_LBL, '*.txt'))):
        name = os.path.splitext(os.path.basename(lp))[0]
        ip = os.path.join(TRAIN_IMG, name + '.jpg')
        if not os.path.exists(ip):
            skipped["no_image"] += 1
            continue
        try:
            w, h = Image.open(ip).size
        except Exception:
            skipped["bad_image"] += 1
            continue
        by_cls = {}
        for line in open(lp, encoding='utf-8'):
            p = line.strip().split()
            if len(p) < 5:
                continue
            c = int(p[0])
            cx, cy, bw, bh = map(float, p[1:5])
            x1, y1 = int(round((cx - bw / 2) * w)), int(round((cy - bh / 2) * h))
            x2, y2 = int(round((cx + bw / 2) * w)), int(round((cy + bh / 2) * h))
            by_cls.setdefault(c, []).append((x1, y1, x2, y2))
        if not by_cls:
            skipped["no_box"] += 1
            continue
        # One sample per (image, class): 95 train images contain 2 classes,
        # 2 contain 3 classes. Each sample asks for ONE named class and
        # lists only that class boxes, so instruction and answer stay aligned.
        for c, boxes in sorted(by_cls.items()):
            cls_name = CLASSES[c]
            uniq = sorted(set(boxes))
            skipped["dup_box"] += len(boxes) - len(uniq)
            ans = '<|object_ref_start|>' + ''.join(
                f'<|object_ref|>{x1},{y1},{x2},{y2}<|/object_ref|>' for x1, y1, x2, y2 in uniq
            ) + '<|object_ref_end|>'
            samples.append({'path': ip, 'cls': cls_name, 'n_boxes': len(uniq),
                            'instruction': rng.choice(TEMPLATES).format(cn=CLS_CN[cls_name]),
                            'answer': ans})
    return samples, skipped

CACHE = os.path.join(WORK, 'samples_cache_v3.json')
if os.path.exists(CACHE):
    with open(CACHE, encoding='utf-8') as f:
        samples = json.load(f)
    skipped = {'cached': True}
    print('loaded cached samples:', len(samples))
else:
    samples, skipped = build_samples()
    with open(CACHE, 'w', encoding='utf-8') as f:
        json.dump(samples, f, ensure_ascii=False)
    print('samples cached to', CACHE)
print('train samples:', len(samples), '| skipped:', skipped)
print('example instruction:', samples[0]['instruction'])
print('example answer:', samples[0]['answer'][:200])

In [ ]:
# ---- Data verification BEFORE training (fail fast, do not burn GPU) ----
import collections
from PIL import ImageDraw
per_cls = collections.Counter(s["cls"] for s in samples)
print('per-class train counts:', dict(per_cls))
assert set(per_cls) == set(CLASSES), "missing class in train split"
assert all(v >= 200 for v in per_cls.values()), "a class has too few samples"
nbox = [s["n_boxes"] for s in samples]
print('boxes/image: mean %.2f max %d | images with 1 box: %d' % (
    sum(nbox) / len(nbox), max(nbox), sum(1 for n in nbox if n == 1)))
# center-spread check on raw labels (no positional shortcut allowed)
cxs, cys = [], []
for lp in glob.glob(os.path.join(TRAIN_LBL, '*.txt')):
    for line in open(lp, encoding='utf-8'):
        p = line.strip().split()
        if len(p) >= 5:
            cxs.append(float(p[1])); cys.append(float(p[2]))
central = sum(1 for x, y in zip(cxs, cys) if 0.4 <= x <= 0.6 and 0.4 <= y <= 0.6) / len(cxs)
print('box centers in central 20%% square: %.1f%% (must be < 25%%)' % (central * 100))
assert central < 0.25, 'boxes are center-clustered - positional shortcut risk'
# save one overlay per class for eyeballing
ov_dir = os.path.join(WORK, 'overlays_v3')
os.makedirs(ov_dir, exist_ok=True)
seen = set()
for s in samples:
    if s['cls'] in seen:
        continue
    seen.add(s['cls'])
    img = Image.open(s["path"]).convert("RGB").resize((400, 400), Image.NEAREST)
    d = ImageDraw.Draw(img)
    import re
    for m in re.finditer(r'<\|object_ref\|>(\d+),(\d+),(\d+),(\d+)<\|/object_ref\|>', s['answer']):
        x1, y1, x2, y2 = (int(m.group(i)) * 2 for i in range(1, 5))
        d.rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=3)
    img.save(os.path.join(ov_dir, s['cls'] + '.png'))
    if len(seen) == 6:
        break
print('overlays saved to', ov_dir, '- eyeball them before trusting the run')
print('DATA CHECKS PASSED')

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = 'Qwen/Qwen3-VL-2B-Instruct'
processor = AutoProcessor.from_pretrained(BASE)
tok = processor.tokenizer

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = Qwen3VLForConditionalGeneration.from_pretrained(BASE, quantization_config=bnb, device_map="cuda")
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
# all-linear covers language + vision-tower + projector linears (same as v2)
peft_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
                      task_type="CAUSAL_LM", target_modules="all-linear")
model = get_peft_model(model, peft_cfg)
model.print_trainable_parameters()
print('model ready')

In [ ]:
import torch
from torch.utils.data import Dataset

ANS_START_ID = tok.convert_tokens_to_ids("<|object_ref_start|>")
MAX_LEN = 2048

class DefectDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        s = self.samples[i]
        img = Image.open(s["path"]).convert("RGB")
        msgs = [{'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': s['instruction']}]},
                {'role': 'assistant', 'content': [{'type': 'text', 'text': s['answer']}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        enc = processor(text=[text], images=[img], return_tensors='pt')
        ids = enc['input_ids'][0]
        attn = enc['attention_mask'][0]
        labels = ids.clone()
        # Loss only on the answer: mask everything before the first <|object_ref_start|> token.
        pos = (ids == ANS_START_ID).nonzero(as_tuple=True)[0]
        assert len(pos) > 0, "answer start token missing in sample %d" % i
        labels[:int(pos[0])] = -100
        mm = enc['mm_token_type_ids'][0] if 'mm_token_type_ids' in enc else None
        if len(ids) > MAX_LEN:
            ids, labels, attn = ids[:MAX_LEN], labels[:MAX_LEN], attn[:MAX_LEN]
            if mm is not None:
                mm = mm[:MAX_LEN]
        out = {'input_ids': ids, 'labels': labels,
               'pixel_values': enc['pixel_values'],
               'image_grid_thw': enc['image_grid_thw'][0],
               'attention_mask': attn}
        # NOTE: pixel_values has NO batch dim - it is already patchified
        # (num_patches, 1536). Do NOT take [0]: that keeps only the first
        # patch and the model sees one patch repeated at all positions.
        # That exact bug was in v2 and is the likely root cause of its
        # mode collapse (format learned from text, no visual signal).
        # image_grid_thw DOES have a leading num_images dim, so [0] is right there.
        if mm is not None:
            out['mm_token_type_ids'] = mm
        return out

def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    input_ids, labels, attn = [], [], []
    for b in batch:
        n = maxlen - len(b["input_ids"])
        input_ids.append(torch.cat([b["input_ids"], torch.full((n,), pad, dtype=torch.long)]))
        labels.append(torch.cat([b["labels"], torch.full((n,), -100, dtype=torch.long)]))
        attn.append(torch.cat([b["attention_mask"], torch.zeros(n, dtype=torch.long)]))
    out = {'input_ids': torch.stack(input_ids), 'labels': torch.stack(labels),
           'attention_mask': torch.stack(attn),
           'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
           'image_grid_thw': torch.stack([b['image_grid_thw'] for b in batch])}
    if 'mm_token_type_ids' in batch[0]:
        mms = []
        for b in batch:
            n = maxlen - len(b["input_ids"])
            mms.append(torch.cat([b["mm_token_type_ids"], torch.zeros(n, dtype=torch.long)]))
        out['mm_token_type_ids'] = torch.stack(mms)
    return out

ds = DefectDataset(samples)
print('dataset size:', len(ds))

In [ ]:
# ---- Masking + EOS self-check: run BEFORE training, not after ----
ex = ds[0]
kept = ex["labels"][ex["labels"] != -100]
masked_n = int((ex["labels"] == -100).sum())
decoded_kept = tok.decode(kept, skip_special_tokens=False)
print('total tokens:', len(ex['input_ids']), '| masked (prompt+image):', masked_n,
      '| kept (answer):', len(kept))
print('kept text starts with:', repr(decoded_kept[:60]))
print('kept text ends with:', repr(decoded_kept[-40:]))
assert decoded_kept.startswith('<|object_ref_start|>'), 'MASKING WRONG: answer does not start with object_ref_start'
assert len(kept) < len(ex['input_ids']) * 0.5, 'MASKING WRONG: kept part too long'
assert decoded_kept.rstrip().endswith('<|im_end|>'), 'EOS MISSING: kept labels do not end with <|im_end|> - model will never learn to stop (v2 failure mode)'
print('masking + EOS OK: loss on answer tokens only, ending with EOS')
# ---- Pixel/grid consistency guard: catches the v2 first-patch bug ----
n_patches = int(ex['pixel_values'].shape[0])
grid_prod = int(ex['image_grid_thw'].prod())
print('pixel_values patches:', n_patches, '| grid t*h*w:', grid_prod,
      '| pixel_values shape:', tuple(ex['pixel_values'].shape))
assert n_patches == grid_prod, 'PIXEL BUG: patch count does not match image_grid_thw'
assert n_patches > 100, 'PIXEL BUG: suspiciously few patches - probably only the first patch survived'
print('pixels OK: full image patches reach the model')

In [ ]:
from transformers import Trainer, TrainingArguments

import inspect
_wanted = dict(
    output_dir=ADAPTER_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="paged_adamw_8bit",
    max_grad_norm=1.0,
    logging_steps=10,
    save_strategy="epoch",
    report_to=[],
    remove_unused_columns=False,
    dataloader_num_workers=0,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
_ok = set(inspect.signature(TrainingArguments.__init__).parameters)
_dropped = sorted(k for k in _wanted if k not in _ok)
args = TrainingArguments(**{k: v for k, v in _wanted.items() if k in _ok})
print('TrainingArguments dropped unsupported kwargs:', _dropped)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collate)
trainer.train()
print('training done')
trainer.save_model(ADAPTER_DIR)
hist = [{'step': h.get('step'), 'loss': h.get('loss')} for h in trainer.state.log_history if 'loss' in h]
print('first loss:', hist[0], '| last loss:', hist[-1])
with open(os.path.join(WORK, 'train_log_v3.json'), 'w', encoding='utf-8') as f:
    json.dump(hist, f, ensure_ascii=False, indent=2)
import glob as _glob
print('adapter files:', sorted(_glob.glob(os.path.join(ADAPTER_DIR, '*'))))
print('TRAINING DONE -> adapter at', ADAPTER_DIR)

## 下一步

1. 在 Kaggle 上把本笔记本的输出（`neudet-adapter-v3`）作为输入数据挂到评估本 `qwen3vl-eval-kaggle-v3-neudet.ipynb`，Run all。
2. 对照同本内测出的基线看三项：格式合规率是否接近 100%、Acc@IoU>=0.5 是否超过基线、是否还有重复框（结果 JSON 里检查每图预测框去重前后数量）。
3. 只有复测确实有提升，才把数字写进简历；写法须注明是 NEU-DET 公开基准上的微调验证。